# Browser Action GRPO Training with Qwen2-0.5B-InstructTrain a small language model to generate correct browser actions using GRPO.- **Model**: Qwen/Qwen2-0.5B-Instruct- **Task**: Browser action generation (navigate, click, type, extract, search, scroll, select)- **Method**: TRL GRPOTrainer with LoRA (rank 32)- **Logging**: Weights & Biases (project: tinker-rl-agentic)- **Hub**: arvindcr4/

In [ ]:
# Cell 1: Install dependencies
!pip install -q trl transformers peft datasets accelerate bitsandbytes wandb huggingface_hub

In [ ]:
# Cell 2: GPU check
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"PyTorch version: {torch.__version__}")

In [ ]:
# Cell 3: Auto-login with pre-configured keys
import os

# Set your API keys here or use Colab secrets
# In Colab: Runtime > Manage secrets, add WANDB_API_KEY and HF_TOKEN
try:
    from google.colab import userdata
    os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except:
    # Fallback: set manually
    os.environ.setdefault('WANDB_API_KEY', 'YOUR_WANDB_API_KEY')
    os.environ.setdefault('HF_TOKEN', 'YOUR_HF_TOKEN')

from huggingface_hub import login
login(token=os.environ['HF_TOKEN'])
import wandb
wandb.login(key=os.environ['WANDB_API_KEY'])

In [ ]:
# Cell 4: Define browser action types and create 30 training examples
import json

BROWSER_ACTIONS = {
    "navigate": {"url": "string"},
    "click": {"selector": "string", "element_description": "string"},
    "type": {"selector": "string", "text": "string"},
    "extract": {"selector": "string", "field": "string"},
    "search": {"query": "string"},
    "scroll": {"direction": "string", "amount": "number"},
    "select": {"selector": "string", "value": "string"}
}

ACTION_DESCRIPTIONS = {
    "navigate": "Navigate to a URL in the browser",
    "click": "Click on an element in the page",
    "type": "Type text into an input field",
    "extract": "Extract data from a page element",
    "search": "Search for something using the search bar",
    "scroll": "Scroll the page in a direction",
    "select": "Select a value from a dropdown"
}

# 30 browser task training examples
TRAINING_DATA = [
    {"prompt": "Go to the Google homepage", "action": "navigate", "args": {"url": "https://www.google.com"}},
    {"prompt": "Open the Wikipedia main page", "action": "navigate", "args": {"url": "https://en.wikipedia.org"}},
    {"prompt": "Navigate to GitHub", "action": "navigate", "args": {"url": "https://github.com"}},
    {"prompt": "Visit the Python documentation website", "action": "navigate", "args": {"url": "https://docs.python.org"}},
    {"prompt": "Go to YouTube", "action": "navigate", "args": {"url": "https://www.youtube.com"}},
    {"prompt": "Click the login button", "action": "click", "args": {"selector": "button#login", "element_description": "Login button"}},
    {"prompt": "Click on the first search result", "action": "click", "args": {"selector": "div.search-result:first-child a", "element_description": "First search result link"}},
    {"prompt": "Click the submit button on the form", "action": "click", "args": {"selector": "button[type='submit']", "element_description": "Form submit button"}},
    {"prompt": "Click the navigation menu hamburger icon", "action": "click", "args": {"selector": "button.menu-toggle", "element_description": "Hamburger menu icon"}},
    {"prompt": "Click the Add to Cart button", "action": "click", "args": {"selector": "button.add-to-cart", "element_description": "Add to Cart button"}},
    {"prompt": "Type my email address into the email field", "action": "type", "args": {"selector": "input[type='email']", "text": "user@example.com"}},
    {"prompt": "Enter the search query machine learning in the search box", "action": "type", "args": {"selector": "input[name='q']", "text": "machine learning"}},
    {"prompt": "Fill in the username field with admin", "action": "type", "args": {"selector": "input#username", "text": "admin"}},
    {"prompt": "Type the password into the password field", "action": "type", "args": {"selector": "input[type='password']", "text": "securepass123"}},
    {"prompt": "Extract the page title", "action": "extract", "args": {"selector": "h1", "field": "textContent"}},
    {"prompt": "Get the price of the product", "action": "extract", "args": {"selector": "span.price", "field": "textContent"}},
    {"prompt": "Extract the author name from the article", "action": "extract", "args": {"selector": "span.author-name", "field": "textContent"}},
    {"prompt": "Get the image URL from the main banner", "action": "extract", "args": {"selector": "img.banner", "field": "src"}},
    {"prompt": "Extract the href from the download link", "action": "extract", "args": {"selector": "a.download-link", "field": "href"}},
    {"prompt": "Search for Python tutorials", "action": "search", "args": {"query": "Python tutorials"}},
    {"prompt": "Search for the best restaurants nearby", "action": "search", "args": {"query": "best restaurants nearby"}},
    {"prompt": "Look up React documentation", "action": "search", "args": {"query": "React documentation"}},
    {"prompt": "Search for machine learning courses", "action": "search", "args": {"query": "machine learning courses"}},
    {"prompt": "Scroll down to see more content", "action": "scroll", "args": {"direction": "down", "amount": 500}},
    {"prompt": "Scroll up to the top of the page", "action": "scroll", "args": {"direction": "up", "amount": 1000}},
    {"prompt": "Scroll down a little bit", "action": "scroll", "args": {"direction": "down", "amount": 200}},
    {"prompt": "Select United States from the country dropdown", "action": "select", "args": {"selector": "select#country", "value": "US"}},
    {"prompt": "Choose English from the language selector", "action": "select", "args": {"selector": "select#language", "value": "en"}},
    {"prompt": "Pick the Large size option", "action": "select", "args": {"selector": "select#size", "value": "large"}},
    {"prompt": "Select the monthly billing plan", "action": "select", "args": {"selector": "select#billing-plan", "value": "monthly"}}
]

ACTIONS_PROMPT = "Available browser actions:\n"
for action_name, params in BROWSER_ACTIONS.items():
    desc = ACTION_DESCRIPTIONS[action_name]
    params_str = json.dumps(params)
    ACTIONS_PROMPT += f"- {action_name}: {desc}. Parameters: {params_str}\n"

print(f"Created {len(TRAINING_DATA)} training examples across {len(BROWSER_ACTIONS)} action types")
print(f"\nAction types: {list(BROWSER_ACTIONS.keys())}")

In [ ]:
# Cell 5: Define reward function for browser actions
import json
import re

def extract_browser_action(text):
    """Extract action name and arguments from model output."""
    try:
        json_match = re.search(r'```json\s*\n?(.*?)\n?```', text, re.DOTALL)
        if json_match:
            parsed = json.loads(json_match.group(1).strip())
            return parsed.get("action") or parsed.get("tool") or parsed.get("name"), parsed.get("args") or parsed.get("arguments") or parsed.get("parameters", {})
        json_match = re.search(r'\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}', text)
        if json_match:
            parsed = json.loads(json_match.group(0))
            return parsed.get("action") or parsed.get("tool") or parsed.get("name"), parsed.get("args") or parsed.get("arguments") or parsed.get("parameters", {})
    except (json.JSONDecodeError, AttributeError):
        pass
    return None, None

def browser_reward_fn(completions, **kwargs):
    """
    Reward function for browser action GRPO.
    - 1.0: correct action type + valid JSON
    - 0.25: valid JSON but wrong action
    - 0.0: invalid output
    """
    prompts = kwargs.get("prompts", [])
    rewards = []
    for i, completion in enumerate(completions):
        if isinstance(completion, list):
            text = completion[-1]["content"] if completion else ""
        else:
            text = completion
        action_name, args = extract_browser_action(text)
        if action_name is None:
            rewards.append(0.0)
            continue
        prompt_text = prompts[i] if i < len(prompts) else ""
        if isinstance(prompt_text, list):
            prompt_text = prompt_text[-1]["content"] if prompt_text else ""
        expected_action = None
        for example in TRAINING_DATA:
            if example["prompt"] in prompt_text:
                expected_action = example["action"]
                break
        if expected_action and action_name == expected_action:
            rewards.append(1.0)
        elif action_name in BROWSER_ACTIONS:
            rewards.append(0.25)
        else:
            rewards.append(0.0)
    return rewards

print("Browser reward function defined.")
print("  1.0  = correct action + valid JSON")
print("  0.25 = valid JSON but wrong action")
print("  0.0  = invalid output")

In [ ]:
# Cell 6: Format dataset for GRPOTrainer
from datasets import Dataset

def format_prompt(example):
    """Format a training example into a chat-style prompt."""
    system_msg = (
        "You are a browser automation assistant. "
        "When the user describes a browser task, respond with a JSON action.\n"
        "Format your response as:\n"
        '```json\n{"action": "<action_name>", "args": {<arguments>}}\n```\n\n'
        f"{ACTIONS_PROMPT}"
    )
    return [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": example["prompt"]}
    ]

formatted_data = []
for example in TRAINING_DATA:
    formatted_data.append({
        "prompt": format_prompt(example),
        "expected_action": example["action"],
        "expected_args": json.dumps(example["args"])
    })

dataset = Dataset.from_list(formatted_data)
print(f"Dataset size: {len(dataset)}")
print(f"Sample prompt: {dataset[0]['prompt'][-1]['content']}")

In [ ]:
# Cell 7: Load model and tokenizer
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True
)

print(f"Model loaded: {MODEL_NAME}")
print(f"Parameters: {model.num_parameters():,}")

In [ ]:
from peft import LoraConfig

# Cell 8: LoRA configuration
lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    task_type="CAUSAL_LM"
)

print(f"LoRA config: rank={lora_config.r}, alpha={lora_config.lora_alpha}")
print(f"Target modules: {lora_config.target_modules}")

In [ ]:
# Cell 9: GRPO configuration
from trl import GRPOConfig

grpo_config = GRPOConfig(
    output_dir="./grpo-browser-qwen0.5b",
    num_train_epochs=5,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=4e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    max_completion_length=256,
    num_generations=16,
    logging_steps=1,
    save_steps=50,
    bf16=True,
    report_to="wandb",
    run_name="browser-grpo-qwen0.5b",
    seed=42,
    remove_unused_columns=False
)

print(f"GRPO Config:")
print(f"  Epochs: {grpo_config.num_train_epochs}")
print(f"  Batch size: {grpo_config.per_device_train_batch_size}")
print(f"  LR: {grpo_config.learning_rate}")
print(f"  Group size: {grpo_config.num_generations}")

In [ ]:
# Cell 10: Create GRPOTrainer and train
from trl import GRPOTrainer

os.environ["WANDB_PROJECT"] = "tinker-rl-agentic"

trainer = GRPOTrainer(
    model=model,
    args=grpo_config,
    train_dataset=dataset,
    reward_funcs=browser_reward_fn,
    peft_config=lora_config,
    processing_class=tokenizer
)

print("Starting GRPO training...")
trainer.train()
print("Training complete!")

In [ ]:
# Cell 11: Save the trained model
OUTPUT_DIR = "./grpo-browser-qwen0.5b-final"
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Model saved to {OUTPUT_DIR}")

In [ ]:
# Cell 12: Evaluation - compare base vs GRPO-trained model
from transformers import pipeline

trained_pipe = pipeline(
    "text-generation",
    model=OUTPUT_DIR,
    tokenizer=tokenizer,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

base_pipe = pipeline(
    "text-generation",
    model=MODEL_NAME,
    tokenizer=tokenizer,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

test_prompts = [
    "Go to the Amazon homepage",
    "Click the sign up button",
    "Type my name into the name field",
    "Extract the product description",
    "Search for laptop deals",
    "Scroll down to the reviews section",
    "Select Express shipping from the dropdown"
]

print("=" * 80)
print("EVALUATION: Base vs GRPO-trained Model (Browser Actions)")
print("=" * 80)

base_correct = 0
trained_correct = 0

for prompt_text in test_prompts:
    messages = format_prompt({"prompt": prompt_text})
    base_out = base_pipe(messages, max_new_tokens=256, do_sample=True, temperature=0.7)
    trained_out = trained_pipe(messages, max_new_tokens=256, do_sample=True, temperature=0.7)
    base_text = base_out[0]["generated_text"][-1]["content"] if isinstance(base_out[0]["generated_text"], list) else base_out[0]["generated_text"]
    trained_text = trained_out[0]["generated_text"][-1]["content"] if isinstance(trained_out[0]["generated_text"], list) else trained_out[0]["generated_text"]
    base_action, _ = extract_browser_action(base_text)
    trained_action, _ = extract_browser_action(trained_text)
    if base_action and base_action in BROWSER_ACTIONS:
        base_correct += 1
    if trained_action and trained_action in BROWSER_ACTIONS:
        trained_correct += 1
    print(f"\nPrompt: {prompt_text}")
    print(f"  Base: {base_action}  |  Trained: {trained_action}")

print(f"\n{'=' * 80}")
print(f"Base accuracy:    {base_correct}/{len(test_prompts)} ({100*base_correct/len(test_prompts):.1f}%)")
print(f"Trained accuracy: {trained_correct}/{len(test_prompts)} ({100*trained_correct/len(test_prompts):.1f}%)")

In [ ]:
# Cell 13: Push to HuggingFace Hub
HUB_REPO = "arvindcr4/browser-grpo-qwen0.5b"

trainer.push_to_hub(HUB_REPO)
tokenizer.push_to_hub(HUB_REPO)

print(f"Model pushed to: https://huggingface.co/{HUB_REPO}")

wandb.finish()
print("Done! W&B run finished.")